# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedwaqasahmad/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The final ranked queue combines my ML-08 Random Forest model score with the three reason codes from my ML-07 baseline, so every recommendation has both a probability (from the model) and a human-readable reason (from the rules) — trust comes from being able to explain why a page is ranked where it is, not just a number.

In [1]:
import os, subprocess
REPO_URL = "https://github.com/syedwaqasahmad/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_churned"] = df["trend_direction"].str.lower().eq("down").astype(int)

numeric_features = ["impressions_90d","search_volume","competition","cpc","word_count","char_count",
                     "days_with_impressions","days_with_sessions","content_age_days",
                     "days_since_last_update","ctr","avg_position","engagement_rate",
                     "scroll_rate","ai_traffic_pct"]
categorical_features = ["competition_level","content_type","main_intent","age_tier",
                         "freshness_tier","word_count_tier","impression_tier","position_tier"]
X_numeric = df[numeric_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_categorical = pd.get_dummies(df[categorical_features], dummy_na=True)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_churned"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=df["client_id"]))
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
model.fit(X.iloc[tr_idx], y.iloc[tr_idx])

df["model_score"] = model.predict_proba(X)[:, 1]
df["reason_stale_visible"] = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(int)
df["reason_declining_demand"] = ((df["trend_direction"].str.lower() == "down") & (df["impressions_90d"] >= 100)).astype(int)
df["reason_position_decay"] = ((df["avg_position"] <= 10) & (df["content_age_days"] >= 180)).astype(int)

def reason_code(row):
    codes = []
    if row["reason_stale_visible"]: codes.append("stale_visible_page")
    if row["reason_position_decay"]: codes.append("page_one_decay_risk")
    if row["model_score"] >= 0.7: codes.append("high_model_risk")
    return ",".join(codes) if codes else "low_priority"

df["reason_code"] = df.apply(reason_code, axis=1)
ranked_queue = df.sort_values("model_score", ascending=False)[
    ["content_id","model_score","reason_code","is_churned","impressions_90d","avg_position"]]
print(ranked_queue.head(10))

                 content_id  model_score                          reason_code  \
16604  content_337ffd45611e          1.0  page_one_decay_risk,high_model_risk   
16571  content_a4a3e160e194          1.0                      high_model_risk   
23727  content_6eeb07cca243          1.0                      high_model_risk   
2260   content_3460f83f024b          1.0                      high_model_risk   
29353  content_a28fe6cce32e          1.0                      high_model_risk   
864    content_edc412bf441c          1.0                      high_model_risk   
16360  content_72c79c304699          1.0                      high_model_risk   
16205  content_00d3b68b4fa7          1.0                      high_model_risk   
17673  content_4044538e77a9          1.0                      high_model_risk   
15704  content_e3d6e71c109e          1.0                      high_model_risk   

       is_churned  impressions_90d  avg_position  
16604           1             7393           8.0  
16571 

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Who uses this: a content/SEO reviewer deciding which pages to check first each week, with limited review capacity.
What it's for: prioritization, not automation — it ranks candidates for human review, it does not auto-publish changes.
Where it stops being valid: it's trained on this one snapshot of 32 clients and 30,000 pages; it hasn't been validated against the full 79M-row warehouse, a different time period, or clients outside this sample. It also should not be used for clients whose content type or industry differs meaningfully from this training set.

In [2]:
print("Trained on:", df["client_id"].nunique(), "clients,", len(df), "pages")
print("Validated: client-holdout split only (ML-08/ML-09). NOT validated on: full warehouse, other time periods, out-of-sample clients.")

Trained on: 32 clients, 30000 pages
Validated: client-holdout split only (ML-08/ML-09). NOT validated on: full warehouse, other time periods, out-of-sample clients.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any recommendation, a reviewer must check: is the decline seasonal/temporary (e.g. a known slow season) rather than structural? Is the traffic drop caused by something outside content quality (a technical issue, an algorithm update, a SERP feature change)?
No-go list — never automate: publishing or unpublishing a page based solely on the model score; treating "high_model_risk" pages as confirmed failures without a human reading the actual page; using this to make personnel or performance judgments about who wrote the content.

In [3]:
print("No-go list:")
print("- Never auto-publish/unpublish based on score alone")
print("- Never treat high_model_risk as a confirmed failure without human review")
print("- Never use this to judge content creators/writers")

No-go list:
- Never auto-publish/unpublish based on score alone
- Never treat high_model_risk as a confirmed failure without human review
- Never use this to judge content creators/writers


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain triggers: if the honest Precision@20/50 (measured monthly on new data) drops meaningfully below the 0.70/0.68 baseline established in ML-08/09, or if the client mix changes substantially (new industries, very different content types), or if 6+ months pass since the last retrain — search behavior and client portfolios drift over time.

In [4]:
print("Retrain triggers:")
print(f"- Precision@20 drops below ~0.70 (current honest baseline) on fresh data")
print(f"- Precision@50 drops below ~0.68 (current honest baseline) on fresh data")
print("- Client portfolio composition changes substantially")
print("- 6+ months since last retrain")

Retrain triggers:
- Precision@20 drops below ~0.70 (current honest baseline) on fresh data
- Precision@50 drops below ~0.68 (current honest baseline) on fresh data
- Client portfolio composition changes substantially
- 6+ months since last retrain


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Writing the final ranked queue to work/outputs/, so my capstone paper can reference these files directly instead of recomputing everything from scratch.

In [5]:
import os as _os
_os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/final_action_queue.csv", index=False)
print("Saved", len(ranked_queue), "rows to work/outputs/final_action_queue.csv")

Saved 30000 rows to work/outputs/final_action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.